# Experimentos #104 — Parte 3: Entrada B (contexto + Top-3 chunks) sobre los datos de #104

A4 (#103, notebook 03) comparó la Entrada A (contexto + título/abstract del citado) con la B (contexto + `perfil : chunk` × Top-3
del citado) con solo **1.241 casos** de entrenamiento etiquetados por Haiku: SciBERT A→B = +0,005, IC95 [−0,021; +0,032]. Los
notebooks 04/05 construyeron ~13k casos por pista (M: `natural` + `kd`; S: `sonnet` + `duro`), pero solo corrieron la Entrada A.
Este notebook entrena modelos nuevos con cada entrada sobre **exactamente las mismas citas, etiquetas, hiperparámetros elegidos con
la misma regla y semillas**, en ambas pistas, y deja las predicciones ciegas listas para el Test Gold. Los ganadores de #104 solo
sirven de referencia (sus `predicciones_val.jsonl`, sin reentrenar).

> **Provisional**: no hay Test Gold humano. V_B sigue siendo fidelidad al maestro (QLoRA/Haiku/Sonnet); decide el Test Gold.

## Diseño (fijado en `preregistro_b_v2.json` antes de cualquier run)

**Estimando.** Primario: **AB − A** (A intacta más los chunks en los tokens sobrantes). Secundario: **B − A**, con una salvedad de
constructo: todas las fuentes de etiqueta vieron solo contexto + título + abstract, es decir, la información de A. B − A mezcla el
aporte de los chunks con la pérdida de la información con la que se formaron las etiquetas; AB − A no la quita.

| Brazo | Definición | Semillas | lr |
|---|---|---|---|
| A | contexto + título + abstract | núcleo | grilla |
| B | contexto + `perfil : chunk` × 3 (Top-3) | núcleo | grilla |
| AB | A intacta + `perfil : chunk` × 3 (Top-3) en los tokens sobrantes | núcleo | grilla |
| AB_aleatorio | A intacta + 3 chunks al azar del mismo citado | control | el de AB |
| B_aleatorio | contexto + los mismos 3 chunks al azar | control | el de B |
| B_barajada | B con los perfiles de sección permutados | control | el de B |
| B_sin_seccion | B sin el prefijo `perfil :` | control | el de B |
| C0 | solo el contexto | control | el de A |
| A-rep | repetición exacta de A-M-s42 | 42 | el de A |
| SPECTER A, B, AB | encoder SPECTER | 42 | el elegido por SciBERT para cada brazo |

- **Semillas.** Núcleo: 42, 1234, 2026, 7, 31337 · control: 42, 1234, 2026 · grilla: 42, 1234. Los chunks al azar usan la sal
  `aleatorio-104-b-v2` + `context_id`.
- **Datos.** train ∩ cobertura B ∩ filtros de fuga. **Dev** propio para la parada temprana: un citante va a dev si
  `sha256("dev-104-b-v2:" + citing_id) % 100 < 15` (la misma regla en M y S). V_B = V ∩ cobertura B solo se evalúa.
  Controles nuevos: L2-dev, L1 sobre {train, dev, V, Gold}, L10 (contextos casi duplicados) y L4 sobre **todos** los chunks de cada citado.
- **Entrenamiento.** Parada temprana sobre dev con F1-macro, 6 épocas como máximo, paciencia 2. Grilla de lr {2e-5, 3e-5, 5e-5} por
  (pista, brazo ∈ {A, B, AB}) con las semillas 42 y 1234; se elige la mayor media de F1-macro de dev (en empate, el lr menor) con la
  grilla completa.

**Inferencia primaria** (2 pruebas: AB − A en M y en S; F1-macro sobre V_B con la etiqueta de la pista, media de 5 semillas).
Con d_j = F1(AB, j) − F1(A, j) y d̄ su media: v_sem = s²(d) / 5; v_art = varianza bootstrap de d̄ con las semillas fijas (5000
réplicas, semilla 2026, remuestreo de artículos citantes, una misma remuestra para semillas y brazos); SE = √(v_sem + v_art);
gl = (v_sem + v_art)² / (v_sem² / 4) (∞ si v_sem = 0); p = 2·(1 − F_t(|d̄| / SE; gl)); IC95 = d̄ ± t_{0,975; gl}·SE.
Holm por pasos sobre las 2 primarias; el TOST con δ = 0,02 forma una familia aparte, también con Holm.
IC descriptivos: bootstrap jerárquico (parejas de semillas y luego artículos) e `ic_cluster` (bootstrap por citante de la media de semillas).

**Regla de decisión**, en este orden: `incompleto` (falta alguna semilla) → `mejora` / `empeora` (p_Holm < 0,05 y al menos 4 de 5
d_j con el signo de d̄; un d_j = 0 no cuenta) → `equivalente` (p_TOST,Holm < 0,05) → `no_concluyente`.

**Poder declarado.** Efecto mínimo detectable con 80 % de poder ≈ 0,023–0,027 (M) y ≈ 0,029–0,034 (S). Un resultado
`no_concluyente` no es evidencia de equivalencia.

**Secundarias** (exploratorias, p sin ajustar, misma prueba t): B − A, AB − B, AB_aleatorio − AB, B_aleatorio − B, B_barajada − B,
B_sin_seccion − B, A − C0 (los controles con las semillas 42, 1234 y 2026); accuracy y F1 ponderado; F1 por clase con IC y
contribución a ΔF1-macro; citado visto / no visto; validación cruzada entre maestros; sesgo de cobertura; no determinismo; truncamiento y SPECTER.

**Desviaciones declaradas.** (1) Las pistas M y S y la grilla de #104 se eligieron con A sobre V. (2) Validez de constructo de las
etiquetas (ver estimando). (3) 1e-5 no está en la grilla. (4) Cualquier flag `CORRER_*` en False.

## Cambios respecto de la v1

| # | v1 | v2 |
|---|---|---|
| 0 | Resultado B − A en F1-macro: M −0,021, IC95 [−0,034; −0,009]; S −0,012, IC95 [−0,030; +0,007] | Se conserva en `evidencias_experimentos_104_b/` y en git; la v2 escribe en carpetas y experimento `_v2` |
| 1 | B reemplaza título y abstract; sin brazo A+B | Brazo AB (A intacta + chunks en el sobrante) como estimando primario |
| 2 | Hiperparámetros de #104 (elegidos con A sobre V, y V_B ⊂ V); 4 épocas con paciencia 3: la parada no podía activarse | Grilla de lr por (pista, brazo) sobre dev; 6 épocas, paciencia 2 |
| 3 | V_B para la parada y para la evaluación | Dev propio por citante para la parada; V_B solo evalúa |
| 4 | Bootstrap solo de artículos; con 3 semillas, «mismo signo» equivale a p = 0,25 | 5 semillas y prueba t con varianza de semillas + artículos; Holm; TOST |
| 5 | Sin control de la relevancia del Top-3 (B_barajada solo permuta perfiles) | AB_aleatorio y B_aleatorio con chunks al azar del mismo citado |
| 6 | Preregistro escrito con el repo sucio y commiteado junto con los resultados; `dirty` contaba archivos sin seguimiento | Gate: preregistro, manifiesto y código en HEAD antes del primer run; `huella_codigo`; `dirty` sin archivos sin seguimiento |
| 7 | Solo F1-macro; F1 por clase sin IC; `afirmable` sin dirección; sin conclusión | Veredicto con dirección y 4 estados; F1 por clase con IC; accuracy y F1 ponderado |
| 8 | Contextos casi duplicados entre artículos gemelos sin control | L10 entre train, dev y V |
| 9 | F1-macro dominada por clases con pocos casos | Contribución por clase a ΔF1-macro, accuracy y F1 ponderado |
| 10 | Las etiquetas vieron solo la información de A | Salvedad de constructo declarada; AB − A como primario |
| 11 | Pistas M y S elegidas con A sobre V | Desviación declarada |
| 12 | Faltan C0 y B sin sección exigidos por el protocolo | Brazos C0 y B_sin_seccion |
| 13 | Reproducibilidad del entrenamiento sin medir | A-rep (repetición de A-M-s42): concordancia y \|ΔF1\| |
| 14 | Sesgo por restringir a la cobertura B sin medir | A del núcleo sobre V completo frente a V_B y frente a los ganadores de #104 |
| 15 | `val_mixta` / `val_sonnet` registradas pero no analizadas | Validación cruzada entre maestros con `predicciones_val.jsonl` |

In [2]:
import argparse
import dataclasses
import gc
import hashlib
import json
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "packages/clasificacion-funcion/src"))
sys.path.insert(0, str(REPO / "packages/auditoria-corpus/src"))
sys.path.insert(0, str(REPO / "packages/recuperacion-densa/src"))
sys.path.insert(0, str(REPO))  # `src.tracking` y `proyecto_de_grado` (namespace desde la raiz)

import mlflow
import pandas as pd
import torch
from auditoria_corpus.dominio import perfil_seccion
from recuperacion_densa.cache_embeddings import CacheDisco

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import cli, cli_modelo, descarga_ocl, dominio
from clasificacion_funcion import entrada_b_104 as eb
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd
from clasificacion_funcion import servicio

In [3]:
DRY = False          # True: <=300 citas por conjunto, 64 casos y 1 epoca por run, menos semillas y lr; experimento y carpetas aparte
DESCARGAR = True     # True: descarga de HuggingFace (ACL-OCL, público; solo ids de artículos) los citados que faltan
HORAS_MAX = 14.0     # límite de tiempo de ejecución; al superarlo, correr() omite los runs restantes
LIM_DRY = 300
L4_DESCARTAR = True            # L4: los citados con chunks que solapan se descartan de train, dev y V; False = abortar
UMBRAL_L4 = eb.UMBRAL_SOLAPE   # 0.5 preregistrado
UMBRAL_L10 = eb.UMBRAL_SOLAPE  # 0.5 preregistrado: contextos casi duplicados entre train, dev y V

# Controles: si alguno queda en False, el preregistro lo declara como desviacion
CORRER_AB_ALEATORIO = True   # AB con 3 chunks al azar del mismo citado
CORRER_B_ALEATORIO = True    # B con los mismos 3 chunks al azar
CORRER_SIN_SECCION = True    # B sin el prefijo `perfil :`
CORRER_BARAJADA = True       # B con los perfiles permutados
CORRER_C0 = True             # solo el contexto
CORRER_REPETICION = True     # A-M-s42 repetido (no determinismo)
CORRER_SPECTER = True        # SPECTER A, B, AB con s42

VERSION = "v2"
SEMILLAS_NUCLEO_PLAN = (42, 1234, 2026, 7, 31337)
SEMILLAS_CONTROL_PLAN = (42, 1234, 2026)
SEMILLAS_GRILLA = (42, 1234)
GRILLA_LR_PLAN = (2e-5, 3e-5, 5e-5)
SEMILLAS_NUCLEO = SEMILLAS_GRILLA if DRY else SEMILLAS_NUCLEO_PLAN
SEMILLAS_CONTROL = (42,) if DRY else SEMILLAS_CONTROL_PLAN
GRILLA_LR = (2e-5, 5e-5) if DRY else GRILLA_LR_PLAN
BRAZOS_GRILLA = ("A", "B", "AB")
# Brazo cuyo lr elegido usa cada entrada
LR_DE = {"A": "A", "B": "B", "AB": "AB", "AB_aleatorio": "AB", "B_aleatorio": "B", "B_barajada": "B",
         "B_sin_seccion": "B", "C0": "A"}
PISTAS = {"M": {"variante": "natural", "modo": "kd", "pista": "mixta"},
          "S": {"variante": "sonnet", "modo": "duro", "pista": "sonnet"}}
HP = dict(epocas=6, paciencia=2)   # el lr sale de la grilla
HP_BASE = ct.Hiperparametros(**HP)
FRACCION_DEV = 0.15
SAL_DEV = "dev-104-b-v2"
SAL_ALEATORIO = "aleatorio-104-b-v2"
SEMILLA_BARAJADO_ESPEC = 42   # la especificacion de B_barajada se registra con s42; cada run baraja con su propia semilla
N_BOOTSTRAP = 5000
SEMILLA_BOOTSTRAP = 2026
DELTA = 0.02
ALFA = 0.05
MIN_SIGNO = len(SEMILLAS_NUCLEO) if DRY else 4
REFERENCIA_104 = {"M": "scibert-A-natural-kd-lr3e-05-e4", "S": "scibert-S-sonnet-lr3e-05-e4"}  # + "-s<semilla>"
SEMILLAS_104 = (42, 1234, 2026)

SUFIJO = "-dry" if DRY else ""
ARTIFACTS = REPO / f"proyecto_de_grado/artifacts/experimentos_104_b_v2{'_dry' if DRY else ''}"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_experimentos_104_b_v2{'_dry' if DRY else ''}"
CORRIDAS = ARTIFACTS / "corridas"
GOLD_DIR = ARTIFACTS / "gold_v2"
DIR_104 = REPO / "proyecto_de_grado/artifacts/experimentos_104/corridas"
EVIDENCIAS.mkdir(parents=True, exist_ok=True)
EXPERIMENTO = rm.EXPERIMENTO_B_V2 + SUFIJO
T0 = time.time()
FALLAS = []
LISTA_RUNS = {}   # run_id -> {nombre, encoder, entrada, pista, semilla, lr, fase, repeticion}

# En DRY los artefactos de MLflow van a una carpeta *_dry (borrable), no a `mlartifacts/` de la raiz
rm.configurar(EXPERIMENTO, artifacts_dir=(ARTIFACTS / "mlartifacts") if DRY else None)
CLIENTE = mlflow.MlflowClient()
print("tracking:", mlflow.get_tracking_uri(), "| experimento:", EXPERIMENTO, "| DRY =", DRY, "| DESCARGAR =", DESCARGAR)


def hay_tiempo() -> bool:
    return (time.time() - T0) / 3600 < HORAS_MAX


def liberar_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def escribir_json(nombre: str, obj) -> Path:
    ruta = EVIDENCIAS / nombre
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return ruta


def git(*args) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)


def correr(nombre, fn, *a, **kw):
    # Ejecuta un run sin detener el notebook: la falla queda FAILED en MLflow y en FALLAS.
    # Ante `OutOfMemoryError` reintenta una vez con micro-lote 4 y acumulacion 4 (mismo lote efectivo).
    if not hay_tiempo():
        print(f"[omitido: límite de tiempo alcanzado] {nombre}")
        return None
    t = time.time()
    for intento in (1, 2):
        try:
            run_id = fn(nombre, *a, **kw)
            print(f"[ok {time.time() - t:6.0f}s] {nombre} -> {run_id}")
            return run_id
        except torch.OutOfMemoryError:
            print(f"[OOM] {nombre} (intento {intento})")
            if intento == 1 and "hp" in kw:
                kw = {**kw, "hp": dataclasses.replace(kw["hp"], batch_size=4, acumulacion=4)}
                continue
            FALLAS.append(nombre)
            return None
        except Exception:
            FALLAS.append(nombre)
            print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
            return None
        finally:
            liberar_gpu()

tracking: sqlite:////home/carlos/microproyecto-local-citation/mlflow.db | experimento: clasificacion-funcion-104-b-v2 | DRY = False | DESCARGAR = True


## Fase 0 — Datos de #104 y controles L1–L3

- **L1** reglas de contexto de #104 (ids, citante, par, texto, años 2014/2015, Gold v2 + TG2700 + taller/piloto/calibración): `sd.construir_todo` falla cerrado.
- **L2**: se descartan de entrenamiento (en todas las entradas, para que usen las mismas citas) las citas cuyo citado es citante de
  V ∪ Gold v2 ∪ TG2700 ∪ excluidos: con chunks, su input llevaría frases de citas de evaluación.
- **L3** ninguna cita de V cita a un citante del Gold.
- **L7** el Test Gold anotado **no se carga nunca**: del Gold solo se leen ids (manifiestos).

In [4]:
FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
CTX = FUENTES.contextos
GOLD_V2_IDS = sorted(ad.FuentesProyectoGrado().ids_gold_v2())  # solo ids del manifiesto (L7)
assert FUENTES.sonnet, "Faltan las etiquetas Sonnet consolidadas: corra la fase 0 del notebook 05"

# L1 (lanza ValueError si hay cualquier fuga)
RES = sd.construir_todo(FUENTES)
assert RES["fugas"]["total_fugas"] == 0 and RES["fugas"]["citas_de_anios_excluidos"] == 0
V, V_SONNET, VAR = RES["v"], RES["v_sonnet"], RES["variantes"]

# Los MISMOS datos que usaron los notebooks 04/05 (si cambian, este analisis ya no seria sobre #104)
M104 = json.loads((REPO / "proyecto_de_grado/docs/evidencias_experimentos_104/manifiesto_datos.json").read_text(encoding="utf-8"))
SHA_ACTUAL = {n: dominio.sha_ids(x["context_id"] for x in filas)
              for n, filas in {"natural": VAR["natural"], "sonnet": VAR["sonnet"], "validacion": V,
                               "validacion_sonnet": V_SONNET}.items()}
assert SHA_ACTUAL == {n: M104["sha256_ids"][n] for n in SHA_ACTUAL}, "Los datos ya no coinciden con #104"

# L2 / L3
CITANTES = eb.citantes_evaluacion(FUENTES, V)
TRAIN_M, QUITADAS_M = eb.excluir_citas_a_citantes(VAR["natural"], CTX, CITANTES)
TRAIN_S, QUITADAS_S = eb.excluir_citas_a_citantes(VAR["sonnet"], CTX, CITANTES)
L3 = eb.verificar_citados_v(V, CTX, eb.citantes_gold(FUENTES))
L1 = sd.verificar({"natural": TRAIN_M, "sonnet": TRAIN_S}, V, FUENTES)   # se vuelve a verificar tras L2
assert L1["total_fugas"] == 0
print(f"L2 quita {sum(QUITADAS_M.values())} de {len(VAR['natural'])} (natural) y {sum(QUITADAS_S.values())} de {len(VAR['sonnet'])} (sonnet)")
print("  natural por clase:", QUITADAS_M)
print("  sonnet  por clase:", QUITADAS_S)
print("L3:", L3)

# Subconjuntos de la prueba en seco (los datos completos ya se verificaron arriba)
def lim(filas):
    return filas[:LIM_DRY] if DRY else filas

TRAIN = {"M": lim(TRAIN_M), "S": lim(TRAIN_S)}
V_USO = lim(V)
_ids_v = {x["context_id"] for x in V_USO}
V_SONNET_USO = [x for x in V_SONNET if x["context_id"] in _ids_v]
GOLD_USO = lim(GOLD_V2_IDS)
print({"train_M": len(TRAIN["M"]), "train_S": len(TRAIN["S"]), "V": len(V_USO), "V_sonnet": len(V_SONNET_USO), "gold_v2": len(GOLD_USO)})

L2 quita 538 de 13331 (natural) y 561 de 13325 (sonnet)
  natural por clase: {'Application': 160, 'Background': 180, 'Basis': 23, 'Comparison': 87, 'Evidence': 37, 'Further Reading': 18, 'Gap': 13, 'Identification of the Originator': 9, 'Improvement / Modification': 11}
  sonnet  por clase: {'Application': 79, 'Background': 117, 'Basis': 27, 'Comparison': 148, 'Evidence': 104, 'Further Reading': 40, 'Gap': 29, 'Identification of the Originator': 4, 'Improvement / Modification': 13}
L3: {'citas_v_a_citantes_gold': 0, 'citas_v': 3379}
{'train_M': 12793, 'train_S': 12764, 'V': 3379, 'V_sonnet': 3379, 'gold_v2': 1533}


## Fase 1 — Citados requeridos y descarga

Las entradas con chunks necesitan el texto completo (ACL-OCL) de cada citado. Con `DESCARGAR = True` se descargan del dataset público
de HuggingFace los citados que faltan en `cache_ocl` (se envían solo ids de artículos ACL). Con `DESCARGAR = False` se usa solo la
caché y se avisa si la cobertura queda por debajo de la esperada.

In [5]:
CONJ = {"train_M": eb.casos_para_chunks(TRAIN["M"], CTX), "train_S": eb.casos_para_chunks(TRAIN["S"], CTX),
        "V": eb.casos_para_chunks(V_USO, CTX), "gold_v2": eb.casos_para_chunks([{"context_id": c} for c in GOLD_USO], CTX)}
REQUERIDOS = sorted({c["cited_id"] for casos in CONJ.values() for c in casos})
CACHE_OCL = ad.CACHE_OCL


def en_cache(cited: str) -> bool:
    return (CACHE_OCL / f"{cited}.json").is_file() or (CACHE_OCL / f"{cited}.404").exists()


FALTAN = [c for c in REQUERIDOS if not en_cache(c)]
print(f"citados requeridos: {len(REQUERIDOS)} | sin descargar: {len(FALTAN)}")
if FALTAN and DESCARGAR:
    lista = EVIDENCIAS / "citados_requeridos_b.txt"
    lista.write_text("".join(f"{c}\n" for c in FALTAN), encoding="utf-8")
    n_previos = len(list(EVIDENCIAS.glob("manifiesto_ocl_b_*.json")))
    rc = descarga_ocl.main(["--citados", str(lista), "--cache", str(CACHE_OCL),
                            "--manifiesto", str(EVIDENCIAS / f"manifiesto_ocl_b_{n_previos + 1}.json")])
    print("descarga_ocl rc =", rc)
    FALTAN = [c for c in REQUERIDOS if not en_cache(c)]
    print("siguen sin descargar:", len(FALTAN))
elif FALTAN:
    print(f"AVISO: DESCARGAR = False y faltan {len(FALTAN)} citados: la cobertura B sera menor a la esperada (~55 %).")

citados requeridos: 3658 | sin descargar: 0


## Fase 2 — Chunks y Top-3 (GPU)

Mismo recuperador que A1/A4 (SciBERT base sin ajustar, revisión de A1) y la misma heurística `perfil_seccion`: ninguno ve etiquetas (**L6**).
Los embeddings se reutilizan de la caché de A4. Reanudable: si `top3_<conjunto>.csv` ya existe con los mismos contextos, no se recalcula.
**Regresión L6**: en los contextos compartidos con los `top3_*.csv` de A4, los `chunk_id` y su orden deben coincidir.

In [6]:
CASOS_CHUNKS, EXCL_CHUNKS = servicio.construir_casos_split(CONJ, CTX, PAPERS, CACHE_OCL, ad.integrar, ad.fragmentar)
print({n: f"{len(c)}/{len(CONJ[n])}" for n, c in CASOS_CHUNKS.items()})

MODELO_RECUPERADOR = cli.EVIDENCIAS / "manifiesto_modelo_scibert.json"
CODIFICADOR, CACHE_EMB, HUELLAS, T_REC = None, CacheDisco(cli.CACHE_EMBEDDINGS), {}, 0.0
TOP3_FILAS = {}
for nombre, casos in CASOS_CHUNKS.items():
    ruta = EVIDENCIAS / f"top3_{nombre}.csv"
    esperados = {c["context_id"] for c in casos}
    if ruta.is_file() and {f["context_id"] for f in cli.leer_csv_top3(ruta)} == esperados:
        TOP3_FILAS[nombre] = cli.leer_csv_top3(ruta)   # reanudacion: la GPU ya hizo este conjunto
        continue
    if not casos:
        TOP3_FILAS[nombre] = []
        continue
    if CODIFICADOR is None:
        CODIFICADOR = cli_modelo.fabrica_recuperador(argparse.Namespace(
            device="auto", batch_size=16, max_length=512, manifiesto_modelo=MODELO_RECUPERADOR))
    t = time.time()
    res = servicio.recuperar_top3(casos, CODIFICADOR, CACHE_EMB, max_length=512,
                                  pooling=getattr(CODIFICADOR, "pooling", "mean"))
    T_REC += time.time() - t
    HUELLAS[nombre] = res["huella"]
    TOP3_FILAS[nombre] = cli._filas_top3(res["top3"])
    ruta.write_text(cli._csv_texto(cli.COL_TOP3, TOP3_FILAS[nombre]), encoding="utf-8", newline="")
    print(f"top3 {nombre}: {len(casos)} contextos, {res['articulos']} articulos, {time.time() - t:.0f}s")


# L6: regresion contra los top3_*.csv de A4 (mismos chunk_id y orden en los contextos compartidos)
def como_top3(filas):
    return {ctx: [{"posicion": r["posicion"], "chunk_id": r["chunk_id"], "similitud_coseno": float(r["puntaje"])}
                  for r in rs] for ctx, rs in servicio.agrupar_top3(filas).items()}


# eval_2015 y gold_v2 de A4 comparten contextos (ambos son test 2015): se unen sin duplicar y se exige que coincidan entre si
_A4 = {}
for n in ("train", "val", "eval_2015", "gold_v2"):
    for f in cli.leer_csv_top3(cli.EVIDENCIAS / f"top3_{n}.csv"):
        previa = _A4.setdefault((f["context_id"], f["posicion"]), f)
        assert (previa["chunk_id"], previa["puntaje"]) == (f["chunk_id"], f["puntaje"]), f"A4 inconsistente consigo mismo: {f['context_id']}"
A4_FILAS = list(_A4.values())
REGRESION = {n: servicio.regresion_top3(como_top3(f), A4_FILAS) for n, f in TOP3_FILAS.items() if f}
for n, r in REGRESION.items():
    print(f"L6 {n}: {r['coinciden']}/{r['contextos_comparados']} contextos comparados con A4, dif max similitud {r['max_dif_similitud']:.1e}")
    assert not r["discrepancias"], f"Regresion contra A4 fallida en {n}: {r['discrepancias'][:5]}"
if not DRY:
    assert REGRESION["gold_v2"]["contextos_comparados"] > 0, "La regresion contra A4 no comparo ningun contexto"

{'train_M': '7003/12793', 'train_S': '6915/12764', 'V': '1818/3379', 'gold_v2': '840/1533'}
L6 train_M: 1144/1144 contextos comparados con A4, dif max similitud 0.0e+00
L6 train_S: 1144/1144 contextos comparados con A4, dif max similitud 0.0e+00
L6 V: 246/246 contextos comparados con A4, dif max similitud 0.0e+00
L6 gold_v2: 840/840 contextos comparados con A4, dif max similitud 0.0e+00


## Fase 3 — Intersección, dev, controles L1–L10 y manifiesto

1. `servicio.interseccion`: un contexto entra si tiene chunks, un Top-3 completo del citado y título/abstract (cobertura ≈55 %).
2. **Dev** por citante (`separar_dev`) sobre el train de cobertura B de cada pista.
3. **L2-dev**: se quitan de train las citas cuyo citado es citante de dev (sus chunks llevarían frases de dev).
4. **L10**: contextos que comparten ≥50 % de shingles de 8 palabras entre train↔dev, train↔V y dev↔V (la misma cita en artículos
   gemelos con distinto id). En train↔dev y train↔V se descarta el caso de train; en dev↔V, el de dev.
5. **L1** con `verificar_fugas` sobre {train, dev, V, Gold} por pista.
6. **L4** sobre **todos** los chunks de cada citado (los controles al azar usan chunks fuera del Top-3): train contra V, Gold y dev;
   dev contra V y Gold; V contra Gold. Con `L4_DESCARTAR = True` los citados con algún chunk que solapa se descartan de train, dev y V
   y L4 se vuelve a correr hasta 0; con `False` el notebook aborta.
7. Entradas con 3 chunks al azar del mismo citado (`entradas_aleatorias`) y su solape con el Top-3.
8. **L5**: todas las entradas usan exactamente los mismos ids que A en train, dev y V.
9. **L9**: en AB y AB_aleatorio los tokens de A son prefijo de los de AB hasta `len(A) − 1 − (a − a′)`, con `a − a′ ≤ 3`
   (los 3 `[SEP]` de los chunks pueden recortar hasta 3 tokens del abstract).
10. `fugas_b.json` (todo bloqueante en 0) y `manifiesto_datos_b.json` (determinista: entra en el preregistro por su sha).

In [7]:
TOP3 = {n: servicio.agrupar_top3(f) for n, f in TOP3_FILAS.items()}
INTER = servicio.interseccion(CONJ, CASOS_CHUNKS, EXCL_CHUNKS, TOP3, PAPERS)
ENTRADAS_B = eb.entradas_por_contexto(INTER["entradas"])
COBERTURA = INTER["resumen"]
for n, r in COBERTURA.items():
    print(f"{n:8s} candidatos={r['candidatos']:6d} interseccion={r['interseccion']:6d} cobertura={r['cobertura_pct']}% excluidos={r['excluidos_por_motivo']}")
print(pd.DataFrame({"candidatos": COBERTURA["train_M"]["por_clase_candidatos"],
                    "con B": COBERTURA["train_M"]["por_clase_interseccion"]})
      .assign(**{"cobertura %": lambda d: (100 * d["con B"] / d["candidatos"]).round(1)}).to_string())


def cubiertas(filas):
    return [x for x in filas if x["context_id"] in ENTRADAS_B]


def ids(filas) -> list[str]:
    return [x["context_id"] for x in filas]


V_B = {"M": cubiertas(V_USO), "S": cubiertas(V_SONNET_USO)}
GOLD_B = [c for c in GOLD_USO if c in ENTRADAS_B]

# Dev para la parada temprana: la misma regla por citante en las dos pistas
TRAIN_B, DEV_B = {}, {}
for k in PISTAS:
    TRAIN_B[k], DEV_B[k] = eb.separar_dev(cubiertas(TRAIN[k]), CTX, FRACCION_DEV, SAL_DEV)

# L2-dev: train no puede citar un articulo citante de dev (sus chunks llevarian frases de dev)
QUITADAS_DEV = {}
for k in PISTAS:
    citantes_dev = {CTX[c]["citing_id"] for c in ids(DEV_B[k])}
    TRAIN_B[k], QUITADAS_DEV[k] = eb.excluir_citas_a_citantes(TRAIN_B[k], CTX, citantes_dev)
    print(f"{k}: train {len(TRAIN_B[k])} | dev {len(DEV_B[k])} | L2-dev quita {sum(QUITADAS_DEV[k].values())} {QUITADAS_DEV[k]}")
print({"V_B": len(V_B["M"]), "V_sonnet_B": len(V_B["S"]), "gold_v2_B": len(GOLD_B)})
assert all(TRAIN_B[k] and DEV_B[k] and V_B[k] for k in PISTAS), "Sin cobertura B: revise la descarga (Fase 1)"

train_M  candidatos= 12793 interseccion=  6996 cobertura=54.69% excluidos={'ValueError': 22, 'autocita': 6, 'menos_de_3_chunks': 7, 'ocl_404': 33, 'parrafo_largo': 5729}
train_S  candidatos= 12764 interseccion=  6908 cobertura=54.12% excluidos={'ValueError': 21, 'autocita': 12, 'menos_de_3_chunks': 7, 'ocl_404': 34, 'parrafo_largo': 5782}
V        candidatos=  3379 interseccion=  1816 cobertura=53.74% excluidos={'ValueError': 6, 'autocita': 4, 'menos_de_3_chunks': 2, 'ocl_404': 3, 'parrafo_largo': 1548}
gold_v2  candidatos=  1533 interseccion=   839 cobertura=54.73% excluidos={'menos_de_3_chunks': 1, 'parrafo_largo': 693}
                                  candidatos  con B  cobertura %
Background                              5174   2858         55.2
Gap                                      435    254         58.4
Basis                                    577    301         52.2
Comparison                              1323    733         55.4
Application                             3538 

In [8]:
def textos(filas) -> dict[str, str]:
    return {x["context_id"]: CTX[x["context_id"]]["masked_text"] for x in filas}


# L10: contextos casi duplicados (articulos gemelos con distinto id) entre train, dev y V
def correr_l10(k):
    filas = (*TRAIN_B[k], *DEV_B[k], *V_B[k])
    return eb.contextos_casi_duplicados({"train": textos(TRAIN_B[k]), "dev": textos(DEV_B[k]), "V": textos(V_B[k])},
                                        UMBRAL_L10, citantes={x["context_id"]: CTX[x["context_id"]]["citing_id"] for x in filas})


L10_INICIAL, L10, L10_DESCARTES = {}, {}, {}
for k in PISTAS:
    L10_INICIAL[k] = correr_l10(k)
    quitar = {"train": set(), "dev": set()}
    for r in L10_INICIAL[k]["pares"].values():   # se descarta el caso del primer conjunto del par
        quitar[r["conjunto_a"]] |= {p["a"] for p in r["detalle"]}
    TRAIN_B[k] = [x for x in TRAIN_B[k] if x["context_id"] not in quitar["train"]]
    DEV_B[k] = [x for x in DEV_B[k] if x["context_id"] not in quitar["dev"]]
    L10_DESCARTES[k] = {n: sorted(v) for n, v in quitar.items()}
    L10[k] = correr_l10(k)
    print(f"L10 {k}: " + ", ".join(f"{p} {r['pares']}" for p, r in L10_INICIAL[k]["pares"].items())
          + f" | descarta train {len(quitar['train'])}, dev {len(quitar['dev'])} | tras descartar: {L10[k]['total_pares']}")

# L1 sobre {train, dev, V, Gold} por pista (contexto, citante, par y texto)
CASOS_GOLD_L1 = eb.casos_para_chunks([{"context_id": c} for c in GOLD_USO], CTX)
L1_DEV = {k: dominio.verificar_fugas({"train": eb.casos_para_chunks(TRAIN_B[k], CTX), "dev": eb.casos_para_chunks(DEV_B[k], CTX),
                                      "V": eb.casos_para_chunks(V_B[k], CTX), "gold": CASOS_GOLD_L1})
          for k in PISTAS}
print("L1 {train, dev, V, gold}:", {k: r["total_fugas"] for k, r in L1_DEV.items()})

# L4: todos los chunks de cada citado (no solo el Top-3) frente a los contextos de evaluacion
CONTEXTOS_GOLD = {c: CTX[c]["masked_text"] for c in sorted(FUENTES.ids_gold) if c in CTX}


def dev_union():
    return list({x["context_id"]: x for k in PISTAS for x in DEV_B[k]}.values())


def correr_l4():
    dev = dev_union()
    a = eb.solapes_texto({"train_M": eb.chunks_de_citados(CASOS_CHUNKS, TRAIN_B["M"], CTX),
                          "train_S": eb.chunks_de_citados(CASOS_CHUNKS, TRAIN_B["S"], CTX),
                          "dev": eb.chunks_de_citados(CASOS_CHUNKS, dev, CTX)},
                         {"V": textos(V), "gold": CONTEXTOS_GOLD, "dev": textos(dev)}, umbral=UMBRAL_L4)
    b = eb.solapes_texto({"V": eb.chunks_de_citados(CASOS_CHUNKS, V_B["M"], CTX)}, {"gold": CONTEXTOS_GOLD}, umbral=UMBRAL_L4)
    return {"umbral": UMBRAL_L4, "k_shingle": eb.K_SHINGLE, "pares": {**a["pares"], **b["pares"]},
            "total_violaciones": a["total_violaciones"] + b["total_violaciones"]}


def mostrar_l4(reporte, titulo):
    for par, r in reporte["pares"].items():
        print(f"{titulo} {par}: {r['chunks_unicos']} chunks, max solape {r['max_solape']}, violaciones {r['violaciones']}, histograma {r['histograma']}")


t = time.time()
L4_INICIAL = correr_l4()
mostrar_l4(L4_INICIAL, "L4")
print(f"L4 en {time.time() - t:.0f}s")
L4_DESCARTES = {"citados": [], "citas_por_clase": {}}
if L4_INICIAL["total_violaciones"] and L4_DESCARTAR:
    contaminados = set().union(*(eb.citados_con_solape(L4_INICIAL, n) for n in ("train_M", "train_S", "dev", "V")))
    L4_DESCARTES["citados"] = sorted(contaminados)
    for k in PISTAS:
        for nombre, dic in (("train", TRAIN_B), ("dev", DEV_B), ("V", V_B)):
            dic[k], quitadas = eb.excluir_citas_a_citantes(dic[k], CTX, contaminados)
            L4_DESCARTES["citas_por_clase"][f"{nombre}:{k}"] = quitadas
    print("L4 descarta", len(contaminados), "citados |", L4_DESCARTES["citas_por_clase"])
    L4 = correr_l4()   # tras descartar: debe quedar en 0
    mostrar_l4(L4, "L4 (tras descartar)")
else:
    L4 = L4_INICIAL
eb.exigir_sin_solapes(L4)
assert all(TRAIN_B[k] and DEV_B[k] and V_B[k] for k in PISTAS), "L4 dejo una pista sin datos"
print({f"{n}_{k}": len(d[k]) for n, d in (("train", TRAIN_B), ("dev", DEV_B), ("V", V_B)) for k in PISTAS})

L10 M: train~dev 4, train~V 5, dev~V 1 | descarta train 7, dev 1 | tras descartar: 0
L10 S: train~dev 24, train~V 3, dev~V 2 | descarta train 21, dev 1 | tras descartar: 0
L1 {train, dev, V, gold}: {'M': 0, 'S': 0}
L4 train_M~V: 44795 chunks, max solape 1.0, violaciones 15, histograma {'(0,0.1]': 242, '(0.1,0.2]': 37, '(0.2,0.3]': 9, '(0.3,0.4]': 4, '(0.4,0.5]': 3, '0': 44487, '>0.5': 13}
L4 train_M~gold: 44795 chunks, max solape 0.7097, violaciones 2, histograma {'(0,0.1]': 188, '(0.1,0.2]': 7, '(0.2,0.3]': 5, '(0.3,0.4]': 2, '(0.4,0.5]': 2, '0': 44590, '>0.5': 1}
L4 train_M~dev: 44795 chunks, max solape 1.0, violaciones 27, histograma {'(0,0.1]': 189, '(0.1,0.2]': 21, '(0.2,0.3]': 10, '(0.3,0.4]': 13, '(0.4,0.5]': 1, '0': 44534, '>0.5': 27}
L4 train_S~V: 43277 chunks, max solape 1.0, violaciones 18, histograma {'(0,0.1]': 243, '(0.1,0.2]': 37, '(0.2,0.3]': 10, '(0.3,0.4]': 5, '(0.4,0.5]': 4, '0': 42963, '>0.5': 15}
L4 train_S~gold: 43277 chunks, max solape 0.9556, violaciones 4, hist

In [9]:
# Entradas con 3 chunks al azar del mismo citado (controles AB_aleatorio y B_aleatorio)
USADOS = sorted({c for k in PISTAS for c in ids((*TRAIN_B[k], *DEV_B[k], *V_B[k]))})
ENTRADAS_ALEATORIAS = eb.entradas_aleatorias(CASOS_CHUNKS, {c: ENTRADAS_B[c] for c in USADOS}, perfil_seccion, SAL_ALEATORIO)
RESUMEN_ALEATORIAS = eb.resumen_aleatorias(ENTRADAS_ALEATORIAS, ENTRADAS_B)
print("chunks al azar:", RESUMEN_ALEATORIAS)
TOP3_SHA = servicio.huella_entradas([ENTRADAS_B[c] for c in USADOS], "B")
ALEATORIAS_SHA = servicio.huella_entradas([ENTRADAS_ALEATORIAS[c] for c in USADOS], "B")


def entradas_de(entrada: str) -> dict:
    return ENTRADAS_ALEATORIAS if entrada in dominio.ENTRADAS_ALEATORIAS else ENTRADAS_B


def top3_sha_de(entrada: str) -> str:
    if entrada not in dominio.ENTRADAS_CON_CHUNKS:
        return ""
    return ALEATORIAS_SHA if entrada in dominio.ENTRADAS_ALEATORIAS else TOP3_SHA


# L5: cada entrada usa exactamente los mismos ids que A (por pista, en train, dev y V)
IDS_SHA = {}
for k in PISTAS:
    for parte, filas in (("train", TRAIN_B[k]), ("dev", DEV_B[k]), ("val", V_B[k])):
        a, _ = ex.casos_entrada("A", filas, FUENTES, PAPERS)
        assert len(a) == len(filas), "A omitio citas sin titulo/abstract: las entradas no usarian las mismas citas"
        IDS_SHA[f"{k}_{parte}"] = dominio.sha_ids(ids(a))
        for entrada in ex.ENTRADAS_104[1:]:
            casos, _ = ex.casos_entrada(entrada, filas, FUENTES, PAPERS, entradas_de(entrada))
            assert eb.verificar_mismos_ids(f"{k}_{parte}:A~{entrada}", a, casos) == IDS_SHA[f"{k}_{parte}"]
print("L5:", {n: s[:12] for n, s in IDS_SHA.items()}, "| pares A~X:", list(ex.ENTRADAS_104[1:]))
SHA_DATOS_B = hashlib.sha256(json.dumps(IDS_SHA, sort_keys=True).encode("utf-8")).hexdigest()

# L9: en AB los tokens de A son prefijo de los de AB hasta len(A) - 1 - (a - a'), con a - a' <= 3
TOK = ct.TokenizadorHF(ct.resolver_encoder("scibert", MODELO_RECUPERADOR))


def ejemplos(entrada: str, filas: list[dict]) -> list[dict]:
    casos, _ = ex.casos_entrada(entrada, filas, FUENTES, PAPERS, entradas_de(entrada), etiqueta=None)
    return ct.preparar_ejemplos([{c: v for c, v in x.items() if c != "_fila"} for x in casos], entrada, TOK, HP_BASE.max_len)


FILAS_USADAS = [{"context_id": c, "citing_id": CTX[c]["citing_id"]} for c in USADOS]
EJ_A = ejemplos("A", FILAS_USADAS)
L9 = {}
for entrada in ("AB", "AB_aleatorio"):
    literales, perdida_max, fallas = 0, 0, []
    for x, y in zip(EJ_A, ejemplos(entrada, FILAS_USADAS), strict=True):
        assert x["context_id"] == y["context_id"]
        perdida = y["truncados"]["abstract"] - x["truncados"]["abstract"]
        corte = len(x["input_ids"]) - 1 - perdida
        if not (0 <= perdida <= 3 and y["input_ids"][:corte] == x["input_ids"][:corte]):
            fallas.append(x["context_id"])
        literales += perdida == 0
        perdida_max = max(perdida_max, perdida)
    L9[entrada] = {"casos": len(EJ_A), "frac_prefijo_literal": literales / len(EJ_A), "perdida_max_abstract": perdida_max,
                   "fallas": len(fallas), "ejemplos_fallas": fallas[:5]}
    print(f"L9 {entrada}: {L9[entrada]}")
assert not any(r["fallas"] for r in L9.values()), L9

chunks al azar: {'contextos': 11643, 'solape_medio': 0.10810501302642501, 'frac_identicos': 0.00515331100231899}
L5: {'M_train': '08673d6c90d6', 'M_dev': '14eae13480bf', 'M_val': '5aea25856686', 'S_train': '4dfb37a9b2af', 'S_dev': 'd2d747d28c11', 'S_val': '5aea25856686'} | pares A~X: ['B', 'B_barajada', 'B_sin_seccion', 'AB', 'B_aleatorio', 'AB_aleatorio', 'C0']
L9 AB: {'casos': 11643, 'frac_prefijo_literal': 0.9822210770419995, 'perdida_max_abstract': 3, 'fallas': 0, 'ejemplos_fallas': []}
L9 AB_aleatorio: {'casos': 11643, 'frac_prefijo_literal': 0.9822210770419995, 'perdida_max_abstract': 3, 'fallas': 0, 'ejemplos_fallas': []}


In [10]:
# L8 (solo reporte): citados compartidos entre train y V; Gold que cita a citantes de V/train
CITADOS_TRAIN = {k: {CTX[c]["refid"] for c in ids(TRAIN_B[k])} for k in PISTAS}
citantes_train = {CTX[c]["citing_id"] for k in PISTAS for c in ids(TRAIN_B[k])}
CITANTES_V_Y_TRAIN = {x["citing_id"] for x in V} | citantes_train
L8 = {"v_con_citado_visto_en_train": {k: sum(CTX[c]["refid"] in CITADOS_TRAIN[k] for c in ids(V_B[k])) for k in PISTAS},
      "v_n": {k: len(V_B[k]) for k in PISTAS},
      "gold_v2_cita_citante_de_v_o_train": sum(CTX[c]["refid"] in CITANTES_V_Y_TRAIN for c in GOLD_USO)}
print("L8:", L8)

BLOQUEANTES = {"L1_fugas_contexto": L1["total_fugas"],
               "L1_dev_fugas": sum(r["total_fugas"] for r in L1_DEV.values()),   # verificar_fugas ya lanzo si > 0
               "L3_citas_v_a_citantes_gold": 0,                                   # verificar_citados_v ya lanzo si > 0
               "L4_violaciones_texto": L4["total_violaciones"],
               "L5_ids_distintos": 0,                                             # verificar_mismos_ids ya lanzo si difieren
               "L6_discrepancias_top3": sum(len(r["discrepancias"]) for r in REGRESION.values()),
               "L9_prefijo_a_en_ab": sum(r["fallas"] for r in L9.values()),
               "L10_casi_duplicados": sum(r["total_pares"] for r in L10.values())}
FUGAS = {"dry": DRY, "bloqueantes": BLOQUEANTES, "L1": L1, "L1_dev": L1_DEV,
         "L2_citas_quitadas_por_clase": {"natural": QUITADAS_M, "sonnet": QUITADAS_S}, "L2_dev_citas_quitadas_por_clase": QUITADAS_DEV,
         "L3": L3, "L4": L4, "L4_inicial": L4_INICIAL, "L4_descartes": L4_DESCARTES, "L5_sha_ids": IDS_SHA, "L6": REGRESION,
         "L7": "El Test Gold anotado no se carga; del Gold v2 solo ids; las predicciones se escriben sin etiquetas",
         "L8_solo_reporte": L8, "L9": L9, "L10": L10, "L10_inicial": L10_INICIAL, "L10_descartes": L10_DESCARTES}
escribir_json("fugas_b.json", FUGAS)
assert sum(BLOQUEANTES.values()) == 0, BLOQUEANTES
print("fugas_b.json: todo bloqueante en 0")

# Manifiesto de datos: determinista (sin fechas ni estado de git), su sha entra en el preregistro
DEV_SHA = {k: dominio.sha_ids(ids(DEV_B[k])) for k in PISTAS}
MANIFIESTO_DATOS = escribir_json("manifiesto_datos_b.json", {
    "dry": DRY, "version": VERSION, "top3_sha": TOP3_SHA, "aleatorias_sha": ALEATORIAS_SHA, "sha256_ids_pistas": IDS_SHA,
    "dev": {"fraccion": FRACCION_DEV, "sal": SAL_DEV, "dev_sha": DEV_SHA, "n": {k: len(DEV_B[k]) for k in PISTAS}},
    "n": {**{f"{n}_{k}_B": len(d[k]) for n, d in (("train", TRAIN_B), ("dev", DEV_B)) for k in PISTAS},
          "V_B": len(V_B["M"]), "V_sonnet_B": len(V_B["S"]), "gold_v2": len(GOLD_USO), "gold_v2_B": len(GOLD_B)},
    "sha256_ids": {**{f"{n}_{k}_B": dominio.sha_ids(ids(d[k])) for n, d in (("train", TRAIN_B), ("dev", DEV_B)) for k in PISTAS},
                   "V_B": dominio.sha_ids(ids(V_B["M"])), "V_sonnet_B": dominio.sha_ids(ids(V_B["S"])), "gold_v2_B": dominio.sha_ids(GOLD_B)},
    "datos_104_sha256_ids": SHA_ACTUAL, "cobertura": COBERTURA,
    "sha256_top3_csv": {n: sd.sha_archivo(EVIDENCIAS / f"top3_{n}.csv") for n in TOP3_FILAS if (EVIDENCIAS / f"top3_{n}.csv").is_file()},
    "aleatorias": {"sal": SAL_ALEATORIO, **RESUMEN_ALEATORIAS},
    "L1_dev_total_fugas": {k: r["total_fugas"] for k, r in L1_DEV.items()},
    "L10": {"umbral": UMBRAL_L10, "descartes": {k: {n: len(v) for n, v in d.items()} for k, d in L10_DESCARTES.items()}},
    "L4": {"umbral": UMBRAL_L4, "citados_descartados": len(L4_DESCARTES["citados"])},
    "hiperparametros": HP, "grilla_lr": GRILLA_LR, "semillas": {"nucleo": SEMILLAS_NUCLEO, "control": SEMILLAS_CONTROL, "grilla": SEMILLAS_GRILLA}})
excl = cli._csv_texto(cli.COL_EXCLUIDOS, INTER["excluidos"])
(EVIDENCIAS / "excluidos_b.csv").write_text(excl, encoding="utf-8", newline="")
escribir_json("manifiesto_recuperacion_b.json", {
    "encoder": CODIFICADOR.info() if CODIFICADOR is not None else "reanudado: top3 ya calculado",
    "sha256_manifiesto_modelo": cli._sha_manifiesto(MODELO_RECUPERADOR), "huellas": HUELLAS, "segundos": round(T_REC, 1),
    "cobertura_chunks": {n: [len(c), len(CONJ[n])] for n, c in CASOS_CHUNKS.items()}, "regresion_a4": REGRESION})

L8: {'v_con_citado_visto_en_train': {'M': 1260, 'S': 1270}, 'v_n': {'M': 1719, 'S': 1719}, 'gold_v2_cita_citante_de_v_o_train': 624}
fugas_b.json: todo bloqueante en 0


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b_v2/manifiesto_recuperacion_b.json')

## Fase 3b — Preregistro y gate

`preregistro_b_v2.json` fija el diseño completo: brazos, sha de la especificación de las 8 entradas, sales, grilla y regla de
elección del lr, δ, α, flags, desviaciones, `dev_sha` por pista, sha de `manifiesto_datos_b.json` y `huella_codigo()` (sha del
paquete y de las celdas de código de este notebook). Si ya existe no se sobrescribe: se recalcula y, si difiere, el notebook se
detiene. El gate exige que el preregistro, el manifiesto y el código estén en HEAD sin cambios antes del primer run; el commit del
preregistro queda en `gate_preregistro.json` y como tag de cada run. En DRY el gate se omite y el preregistro se reescribe.

In [11]:
FLAGS = {"CORRER_AB_ALEATORIO": CORRER_AB_ALEATORIO, "CORRER_B_ALEATORIO": CORRER_B_ALEATORIO,
         "CORRER_SIN_SECCION": CORRER_SIN_SECCION, "CORRER_BARAJADA": CORRER_BARAJADA, "CORRER_C0": CORRER_C0,
         "CORRER_REPETICION": CORRER_REPETICION, "CORRER_SPECTER": CORRER_SPECTER}
DESVIACIONES = ["Las pistas M y S y la grilla de #104 se eligieron con A sobre V.",
                "Validez de constructo: todas las fuentes de etiqueta vieron solo contexto + titulo + abstract (la informacion de A).",
                "1e-5 no esta en la grilla de lr."] + [f"{n} = False: ese control no se corre." for n, v in FLAGS.items() if not v]
ESPEC_SHA = {e: dominio.sha_especificacion(dominio.especificacion_entrada(
                 e, HP_BASE.max_len, SEMILLA_BARAJADO_ESPEC if e == "B_barajada" else None,
                 SAL_ALEATORIO if e in dominio.ENTRADAS_ALEATORIAS else None))
             for e in ex.ENTRADAS_104}
DISENO = json.loads(json.dumps({
    "version": VERSION, "dry": DRY, "experimento_mlflow": EXPERIMENTO,
    "estimando": {"primario": "AB - A", "secundario": "B - A (salvedad de constructo)"},
    "brazos": {"nucleo": BRAZOS_GRILLA, "controles": ["AB_aleatorio", "B_aleatorio", "B_sin_seccion", "B_barajada", "C0", "A-rep"],
               "specter": BRAZOS_GRILLA, "lr_de": LR_DE},
    "pistas": PISTAS,
    "semillas": {"nucleo": SEMILLAS_NUCLEO, "control": SEMILLAS_CONTROL, "grilla": SEMILLAS_GRILLA, "specter": [42], "repeticion": [42]},
    "hiperparametros": {k: v for k, v in dataclasses.asdict(HP_BASE).items() if k != "lr"},
    "grilla_lr": GRILLA_LR, "regla_lr": "mayor media de F1-macro de dev entre las semillas de la grilla; empate: lr menor; grilla completa",
    "parada": "F1-macro sobre dev", "dev": {"fraccion": FRACCION_DEV, "sal": SAL_DEV, "regla": "sha256(sal:citing_id) % 100 < 15"},
    "sales": {"dev": SAL_DEV, "aleatorio": SAL_ALEATORIO}, "semilla_barajado_especificacion": SEMILLA_BARAJADO_ESPEC,
    "especificacion_sha": ESPEC_SHA,
    "inferencia": {"primarias": "AB - A por pista; F1-macro sobre V_B con la etiqueta de la pista; media de las semillas del nucleo",
                   "prueba": "t con varianza combinada: v_sem = s2(d)/n_semillas; v_art = var bootstrap por citante con semillas fijas "
                             "(remuestra compartida); gl de Satterthwaite (inf si v_sem = 0)",
                   "n_bootstrap": N_BOOTSTRAP, "semilla_bootstrap": SEMILLA_BOOTSTRAP,
                   "holm": "por pasos sobre las 2 primarias; TOST en familia aparte con Holm",
                   "delta": DELTA, "alfa": ALFA, "min_signo": MIN_SIGNO,
                   "veredicto": ["incompleto", "mejora/empeora", "equivalente", "no_concluyente"],
                   "ic_descriptivos": ["bootstrap jerarquico (semillas y articulos)", "ic_cluster (bootstrap por citante)"]},
    "poder_declarado": {"M": [0.023, 0.027], "S": [0.029, 0.034], "potencia": 0.8},
    "secundarias": ["B - A", "AB - B", "AB_aleatorio - AB", "B_aleatorio - B", "B_barajada - B", "B_sin_seccion - B", "A - C0",
                    "accuracy y F1 ponderado", "F1 por clase con IC y contribucion", "citado visto / no visto",
                    "validacion cruzada entre maestros", "sesgo de cobertura", "no determinismo", "truncamiento", "SPECTER"],
    "test_gold": "predicciones ciegas de A (cobertura y todo), B y AB (cobertura) de los runs del nucleo y SPECTER; B - A y AB - A por pista y semilla",
    "flags": FLAGS, "desviaciones": DESVIACIONES,
    "L4": {"umbral": UMBRAL_L4, "k_shingle": eb.K_SHINGLE, "descartar": L4_DESCARTAR}, "L10": {"umbral": UMBRAL_L10},
    "dev_sha": DEV_SHA, "manifiesto_datos_sha256": sd.sha_archivo(MANIFIESTO_DATOS),
    "huella_codigo": rm.huella_codigo(),
}, default=str))

PRE = EVIDENCIAS / "preregistro_b_v2.json"
if PRE.exists() and not DRY:
    if json.loads(PRE.read_text(encoding="utf-8"))["diseno"] != DISENO:
        raise RuntimeError("el preregistro existente no coincide con el diseño actual")
else:
    escribir_json(PRE.name, {"escrito_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "diseno": DISENO})
print("preregistro:", PRE.relative_to(REPO), "| huella:", {k: v[:12] for k, v in DISENO["huella_codigo"].items()})


def comprobar_codigo(huella_esperada: dict) -> None:
    # El preregistro, el manifiesto y el codigo deben estar en HEAD sin cambios
    fallas = []
    for ruta in (PRE, MANIFIESTO_DATOS):
        rel = str(ruta.relative_to(REPO))
        if git("ls-files", "--error-unmatch", rel).returncode:
            fallas.append(f"ls-files: {rel}")
        elif git("diff", "--quiet", "HEAD", "--", rel).returncode:
            fallas.append(f"diff HEAD: {rel}")
    if git("diff", "--quiet", "HEAD", "--", str(rm.SRC_PAQUETE)).returncode:
        fallas.append(f"diff HEAD: {rm.SRC_PAQUETE}")
    actual = rm.huella_codigo()
    en_head = git("show", f"HEAD:{rm.NOTEBOOK_06}")
    celdas_head = rm.sha_celdas(en_head.stdout) if en_head.returncode == 0 else None
    if actual != huella_esperada or actual["celdas_sha"] != celdas_head:
        fallas.append("huella_codigo")
    if fallas:
        raise RuntimeError("el preregistro, el manifiesto o el código difieren de HEAD: " + "; ".join(fallas))


if DRY:
    GATE = {"commit": "dry", "fecha": None}
    print("DRY: gate omitido")
else:
    comprobar_codigo(DISENO["huella_codigo"])
    commit, fecha = git("log", "-1", "--format=%H%n%cI", "--", str(PRE.relative_to(REPO))).stdout.split()
    GATE = {"commit": commit, "fecha": fecha, "huella_codigo": DISENO["huella_codigo"]}
    escribir_json("gate_preregistro.json", GATE)
    print("gate superado:", GATE)

preregistro: proyecto_de_grado/docs/evidencias_experimentos_104_b_v2/preregistro_b_v2.json | huella: {'src_sha': '01f405dd62e7', 'celdas_sha': 'ff8b7d3b21af'}
gate superado: {'commit': 'c4a536540887832f25a7a06f65711f61e414c25b', 'fecha': '2026-11-14T21:54:50-05:00', 'huella_codigo': {'src_sha': '01f405dd62e7281d7393fb7ce58dd754abaa15e08fd845dc1724fd2d2e298ae5', 'celdas_sha': 'ff8b7d3b21af1ac315b4fb63f506b65159c026e58689c7f052b914d5011632b4'}}


## Fase 4 — Grilla de lr y núcleo

**Grilla**: 3 lr × {A, B, AB} × 2 pistas × semillas {42, 1234}, con parada temprana sobre dev. `seleccionar_lr` elige por
(pista, brazo) la mayor media de F1-macro de dev entre las dos semillas (en empate, el lr menor) y exige la grilla completa:
si falta una combinación, el notebook se detiene. **Núcleo**: semillas 2026, 7 y 31337 × 2 pistas × {A, B, AB} con el lr elegido;
los runs de 42 y 1234 con ese lr vienen de la grilla y se marcan `fase = nucleo`. Todo se reanuda por `config_sha`.

In [12]:
def config_pista(k: str) -> dict:
    p = PISTAS[k]
    cfg = {"diseno": "104-b-v2", "clave_pista": k, "pista": p["pista"], "variante": p["variante"],
           "v_sha": IDS_SHA[f"{k}_val"], "ids_train_sha": IDS_SHA[f"{k}_train"], "cobertura": "interseccion_B"}
    return {**cfg, "v_etiquetas": "sonnet"} if k == "S" else cfg


def run(k: str, entrada: str, semilla: int, lr: float, encoder: str = "scibert", fase: str = "nucleo", repeticion: int = 0):
    nombre = ex.nombre_run(encoder, entrada, k, lr, semilla, repeticion)
    config = {**config_pista(k), **({"repeticion": repeticion} if repeticion else {})}
    run_id = correr(nombre, ex.entrenar_y_registrar, config, TRAIN_B[k], V_B[k], FUENTES, PAPERS, CORRIDAS,
                    encoder=encoder, semilla=semilla, modo=PISTAS[k]["modo"], hp=dataclasses.replace(HP_BASE, lr=lr), dry=DRY,
                    manifiesto_datos_sha=SHA_DATOS_B, entrada=entrada, entradas_b=entradas_de(entrada),
                    top3_sha=top3_sha_de(entrada), dev_filas=DEV_B[k],
                    tags_extra={"preregistro_commit": GATE["commit"], "fase": fase, "repeticion": repeticion})
    if run_id is None:
        return None
    if not (CORRIDAS / nombre / "predicciones_val.jsonl").is_file():
        FALLAS.append(f"sin-predicciones-{nombre}")
        print(f"[FALLO] {nombre}: falta predicciones_val.jsonl")
        return None
    previa = LISTA_RUNS.get(run_id, {}).get("fase")
    fase = "nucleo" if "nucleo" in (fase, previa) else fase
    if fase == "nucleo":   # los runs de la grilla con el lr elegido pasan al nucleo
        CLIENTE.set_tag(run_id, "fase", "nucleo")
    LISTA_RUNS[run_id] = {"nombre": nombre, "encoder": encoder, "entrada": entrada, "pista": k, "semilla": semilla,
                          "lr": float(lr), "fase": fase, "repeticion": repeticion}
    return run_id


for s in SEMILLAS_GRILLA:
    for k in PISTAS:
        for brazo in BRAZOS_GRILLA:
            for lr in GRILLA_LR:
                run(k, brazo, s, lr, fase="grilla")
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

MANIFIESTOS_GRILLA = {}
for k in PISTAS:
    for brazo in BRAZOS_GRILLA:
        for lr in GRILLA_LR:
            for s in SEMILLAS_GRILLA:
                ruta = CORRIDAS / ex.nombre_run("scibert", brazo, k, lr, s) / "manifiesto_entrenamiento.json"
                if ruta.is_file():
                    MANIFIESTOS_GRILLA[(k, brazo, float(lr), s)] = json.loads(ruta.read_text(encoding="utf-8"))
SELECCION = ex.seleccionar_lr(MANIFIESTOS_GRILLA, GRILLA_LR, SEMILLAS_GRILLA,
                              combinaciones=[(k, b) for k in PISTAS for b in BRAZOS_GRILLA])
LR = {clave: r["lr"] for clave, r in SELECCION.items()}
escribir_json("seleccion_lr.json", {
    f"{k}:{b}": {"lr": r["lr"], "medias": {f"{lr:g}": m for lr, m in r["medias"].items()},
                 "por_semilla": {f"{lr:g}": {str(s): v for s, v in ps.items()} for lr, ps in r["por_semilla"].items()}}
    for (k, b), r in SELECCION.items()})
print(pd.DataFrame({f"{k}:{b}": {f"{lr:g}": round(m, 4) for lr, m in r["medias"].items()} | {"elegido": f"{r['lr']:g}"}
                    for (k, b), r in SELECCION.items()}).to_string())

[ok      0s] scibert-A-M-lr2e-05-s42 -> b108cb822aa443ebae7f8d6eed3ca1d0
[ok      0s] scibert-A-M-lr3e-05-s42 -> 6c032667a3344cc1aec4a39d673dbace
[ok      0s] scibert-A-M-lr5e-05-s42 -> 29890ed6acee4df49349da7ceeb1fba9
[ok      0s] scibert-B-M-lr2e-05-s42 -> 1b9739a19f7f4024806f3bf6e931f668
[ok      0s] scibert-B-M-lr3e-05-s42 -> 1680fa211c8047fa8aacd1f4eea54ed4
[ok      0s] scibert-B-M-lr5e-05-s42 -> c9b2a59147954804b4b1d705e908021b
[ok      0s] scibert-AB-M-lr2e-05-s42 -> c66794ae425b43c9a700d24a1201102c
[ok      0s] scibert-AB-M-lr3e-05-s42 -> 0329d2b4efdb451b886ca3e8c2d1a326
[ok      0s] scibert-AB-M-lr5e-05-s42 -> c423d763995d4dfc8ff94368827f9fc7
[ok      0s] scibert-A-S-lr2e-05-s42 -> 723ae5862fc6432bb8ca1b59b8bb99e5
[ok      0s] scibert-A-S-lr3e-05-s42 -> bac30e6bb5ec416f980a0ca058922ab5
[ok      0s] scibert-A-S-lr5e-05-s42 -> 518f8b5ed6f84ca69cb92a43eb7d3ecc
[ok      0s] scibert-B-S-lr2e-05-s42 -> 2f9bd8be2f204bf7b05b081442700724
[ok      0s] scibert-B-S-lr3e-05-s42 -> 2fce8e28

In [13]:
for s in SEMILLAS_NUCLEO:
    for k in PISTAS:
        for brazo in BRAZOS_GRILLA:
            run(k, brazo, s, LR[(k, brazo)], fase="nucleo")
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

[ok      0s] scibert-A-M-lr2e-05-s42 -> b108cb822aa443ebae7f8d6eed3ca1d0
[ok      0s] scibert-B-M-lr2e-05-s42 -> 1b9739a19f7f4024806f3bf6e931f668
[ok      0s] scibert-AB-M-lr2e-05-s42 -> c66794ae425b43c9a700d24a1201102c
[ok      0s] scibert-A-S-lr2e-05-s42 -> 723ae5862fc6432bb8ca1b59b8bb99e5
[ok      0s] scibert-B-S-lr5e-05-s42 -> a8fe9beffcfc4b7ba51261b1722b8f27
[ok      0s] scibert-AB-S-lr3e-05-s42 -> 947a3fc986334c24a236a6889a6f1f90
[ok      0s] scibert-A-M-lr2e-05-s1234 -> 4b71b29927d44a6f9b9d6a7632b03948
[ok      0s] scibert-B-M-lr2e-05-s1234 -> de07928169af46aa9fa8b30ef2a8a0bd
[ok      0s] scibert-AB-M-lr2e-05-s1234 -> efec5352541e4a7692e20993560dc8b3
[ok      0s] scibert-A-S-lr2e-05-s1234 -> cbb4524374b5464f8483dbc3b07de5ca
[ok      0s] scibert-B-S-lr5e-05-s1234 -> 62fedaa34a0f43288da81d12adc6134c
[ok      0s] scibert-AB-S-lr3e-05-s1234 -> b534cdbec0ec4ab1b4a632c8870379be
[ok      0s] scibert-A-M-lr2e-05-s2026 -> e23dcb331a3e4b22b7064eaba15805d0
[ok      0s] scibert-B-M-lr2e-05-

## Fase 5 — Referencia: los ganadores de #104 sobre V_B (cota superior: elegidos sobre V)

Entrada A entrenada con todo el train de #104, sin reentrenar y sin input B: sus `predicciones_val.jsonl` restringidas a V_B. Sus
hiperparámetros se eligieron sobre V, que contiene a V_B: es una cota superior, no un brazo comparable.

In [14]:
ETIQ_V = {k: {x["context_id"]: dominio.normalizar_etiqueta(x["etiqueta_entrenamiento"]) for x in V_B[k]} for k in PISTAS}
V_COMPLETO = {"M": V_USO, "S": V_SONNET_USO}
ETIQ_VC = {k: {x["context_id"]: dominio.normalizar_etiqueta(x["etiqueta_entrenamiento"]) for x in V_COMPLETO[k]} for k in PISTAS}


def f1_en(preds: dict, etiquetas: dict) -> tuple[float, int]:
    comunes = sorted(set(preds) & set(etiquetas))
    return dominio.metricas([etiquetas[c] for c in comunes], [preds[c] for c in comunes])["f1_macro"], len(comunes)


def preds_val(nombre: str, base: Path = CORRIDAS):
    carpeta = base / nombre
    return ex.leer_predicciones(carpeta) if (carpeta / "predicciones_val.jsonl").is_file() else None


REFERENCIA = {}
for k, stem in REFERENCIA_104.items():
    for s in SEMILLAS_104:
        p = preds_val(f"{stem}-s{s}", DIR_104)
        if p:
            REFERENCIA[f"{k}-s{s}"] = dict(zip(("f1_macro_V_B", "n_V_B"), f1_en(p, ETIQ_V[k]))) | \
                dict(zip(("f1_macro_V", "n_V"), f1_en(p, ETIQ_VC[k])))
print("cota superior (elegidos sobre V):")
print(pd.DataFrame(REFERENCIA).T.round(4).to_string() if REFERENCIA else "sin predicciones de los ganadores de #104")

cota superior (elegidos sobre V):
         f1_macro_V_B   n_V_B  f1_macro_V     n_V
M-s42          0.7176  1719.0      0.7167  3379.0
M-s1234        0.7096  1719.0      0.7130  3379.0
M-s2026        0.7013  1719.0      0.7041  3379.0
S-s42          0.6442  1719.0      0.6736  3379.0
S-s1234        0.6521  1719.0      0.6752  3379.0
S-s2026        0.6130  1719.0      0.6513  3379.0


## Fase 5b — A del núcleo sobre V completo

Cada run A del núcleo se evalúa también sobre V completo (sin restringir a la cobertura B), con la etiqueta de su pista. Se
registra como `vcompleto_*` en el mismo run y en `predicciones_vcompleto.jsonl`: mide el sesgo de restringir a la cobertura B.

In [15]:
for run_id, r in [(i, r) for i, r in LISTA_RUNS.items() if r["fase"] == "nucleo" and r["entrada"] == "A"]:
    try:
        m = ex.evaluar_conjunto_extra(run_id, CORRIDAS / r["nombre"], r["encoder"], "vcompleto", V_COMPLETO[r["pista"]],
                                      FUENTES, PAPERS)
        print(f"[vcompleto] {r['nombre']}: " + (f"F1-macro {m['f1_macro']:.4f} (n={m['n']})" if m else "ya registrado"))
    except Exception:
        FALLAS.append(f"vcompleto-{r['nombre']}")
        print(f"[FALLO] vcompleto {r['nombre']}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()

[vcompleto] scibert-A-M-lr2e-05-s42: ya registrado
[vcompleto] scibert-A-S-lr2e-05-s42: ya registrado
[vcompleto] scibert-A-M-lr2e-05-s1234: ya registrado
[vcompleto] scibert-A-S-lr2e-05-s1234: ya registrado
[vcompleto] scibert-A-M-lr2e-05-s2026: ya registrado
[vcompleto] scibert-A-S-lr2e-05-s2026: ya registrado
[vcompleto] scibert-A-M-lr2e-05-s7: ya registrado
[vcompleto] scibert-A-S-lr2e-05-s7: ya registrado
[vcompleto] scibert-A-M-lr2e-05-s31337: ya registrado
[vcompleto] scibert-A-S-lr2e-05-s31337: ya registrado


## Fase 6 — Controles

| Control | Qué mide |
|---|---|
| AB_aleatorio − AB | si los chunks del recuperador aportan más que 3 chunks cualesquiera del mismo citado (relevancia del Top-3) |
| B_aleatorio − B | lo mismo sin título ni abstract |
| B_sin_seccion − B | el aporte del prefijo `perfil :` |
| B_barajada − B | si el perfil de sección correcto aporta señal (permuta los perfiles entre los chunks del conjunto) |
| A − C0 | el aporte de la información del citado sobre el contexto solo |
| A-rep | el no determinismo del entrenamiento: la misma configuración A-M-s42 entrenada otra vez |
| SPECTER A, B, AB | el 2×2 de #103 con el otro encoder (descriptivo, semilla 42) |

Los controles usan las semillas 42, 1234 y 2026 y el lr elegido para su brazo de referencia.

In [16]:
CONTROLES = (("AB_aleatorio", CORRER_AB_ALEATORIO), ("B_aleatorio", CORRER_B_ALEATORIO), ("B_sin_seccion", CORRER_SIN_SECCION),
             ("B_barajada", CORRER_BARAJADA), ("C0", CORRER_C0))
for entrada, activo in CONTROLES:
    if not activo:
        continue
    for s in SEMILLAS_CONTROL:
        for k in PISTAS:
            run(k, entrada, s, LR[(k, LR_DE[entrada])], fase="control")
if CORRER_REPETICION:
    run("M", "A", 42, LR[("M", "A")], fase="control", repeticion=1)
if CORRER_SPECTER:
    for k in PISTAS:
        for brazo in BRAZOS_GRILLA:
            run(k, brazo, 42, LR[(k, brazo)], encoder="specter", fase="specter")
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

[ok      0s] scibert-AB_aleatorio-M-lr2e-05-s42 -> 81b49c9bd8164addb0364d669368aa16
[ok      0s] scibert-AB_aleatorio-S-lr3e-05-s42 -> 1c6f3205fc664d1db2d10d414a1de331
[ok      0s] scibert-AB_aleatorio-M-lr2e-05-s1234 -> 6223f2c870c34c598cfe5c15fb298b57
[ok      0s] scibert-AB_aleatorio-S-lr3e-05-s1234 -> 74ea60e4fcd44826bf4ee8ebb9b6715a
[ok      0s] scibert-AB_aleatorio-M-lr2e-05-s2026 -> 2fe1ab97a95e48d4b80fb1b1a7b6636a
[ok      0s] scibert-AB_aleatorio-S-lr3e-05-s2026 -> ed2d9f7a14ed45618812c4057d2eeb4e
[ok      0s] scibert-B_aleatorio-M-lr2e-05-s42 -> b07ac3643a824c4f98e4a6fc60a0a2e2
[ok      0s] scibert-B_aleatorio-S-lr5e-05-s42 -> e5333aa42ba64424a4a8651a61c815f5
[ok      0s] scibert-B_aleatorio-M-lr2e-05-s1234 -> ae3c4e10d65042e6af6c3fb5a1881276
[ok      0s] scibert-B_aleatorio-S-lr5e-05-s1234 -> bca93771d9e0464dbefb392de07a127d
[ok      0s] scibert-B_aleatorio-M-lr2e-05-s2026 -> db720fec4aa94ffb8ca9a561c70b4228
[ok      0s] scibert-B_aleatorio-S-lr5e-05-s2026 -> ad107da24bf041f

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48129.77it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    768s] scibert-B_barajada-M-lr2e-05-s42 -> e6c7d753d73e440a969050c0f4e3a956


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 60034.99it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    484s] scibert-B_barajada-S-lr5e-05-s42 -> 8f99ac13562346809298bc9ab534ccb1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21510.36it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

... (29 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21534.78it/s]


[ok    727s] specter-B-S-lr5e-05-s42 -> 4d8f2c7f707c42cba223e7d2d46f89cf


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51341.98it/s]


[ok    736s] specter-AB-S-lr3e-05-s42 -> 7e36fa88ed6c4f1396c85ab31e154dad
fallas: ninguna | 2.78 h


## Fase 6b — Smoke test de inferencia

Un checkpoint B, B_barajada y AB por pista (semilla 42): se recarga verificando su sha256 y debe reproducir las predicciones de `val`
guardadas al entrenar.

In [17]:
SMOKE = []
for k in PISTAS:
    for entrada in ("B", "B_barajada", "AB"):
        carpeta = CORRIDAS / ex.nombre_run("scibert", entrada, k, LR[(k, LR_DE[entrada])], 42)
        if not (carpeta / "model.safetensors").is_file():
            continue
        try:
            r = ex.smoke_inferencia(carpeta, "scibert", V_B[k], FUENTES, PAPERS, entrada=entrada, entradas_b=ENTRADAS_B)
        except Exception:
            FALLAS.append(f"smoke-{k}-{entrada}")
            r = {"ok": False, "error": traceback.format_exc()}
        SMOKE.append({"pista": k, "entrada": entrada, **r})
        print({c: v for c, v in SMOKE[-1].items() if c != "error"})
        liberar_gpu()
escribir_json("smoke_inferencia_b.json", SMOKE)
assert SMOKE and all(r["ok"] for r in SMOKE) or not hay_tiempo(), "El smoke test no reproduce las predicciones guardadas"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23292.59it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'M', 'entrada': 'B', 'run': 'scibert-B-M-lr2e-05-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 10.451660937519591, 'checkpoint_sha256': 'f51977779ced65703d630c33a5c8bdbefaa3f9c7e095c37713e8613bfa6f74b8'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24001.91it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'M', 'entrada': 'B_barajada', 'run': 'scibert-B_barajada-M-lr2e-05-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 9.436067656281466, 'checkpoint_sha256': '6fecb7b1c267ea217fcbadf49f97cdbfc417640330154d63f0205f7e9ea9afc9'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 43294.08it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'M', 'entrada': 'AB', 'run': 'scibert-AB-M-lr2e-05-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 6.978695406246516, 'checkpoint_sha256': '071ff94fd201cb39e5741da54f9af03f002629ccb0316733dfd97d971c77f63a'}


... (2 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 67084.59it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'S', 'entrada': 'B_barajada', 'run': 'scibert-B_barajada-S-lr5e-05-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 11.201682999967488, 'checkpoint_sha256': '141420a05c044f1ca6043aff52ca624c551a18db3049b1eeddb63db80ef3d571'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15721.73it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'S', 'entrada': 'AB', 'run': 'scibert-AB-S-lr3e-05-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 11.965390687521449, 'checkpoint_sha256': 'd7a68920447f822d388fcd30b5229bd8328c8f41b1c069690b1ed2c3ab5bc861'}


## Fase 7 — Predicciones ciegas sobre el Gold v2 (sin leer etiquetas)

Solo los runs del núcleo y de SPECTER. A sobre Gold v2 ∩ cobertura (conjunto pareado) y sobre todo el Gold v2; B y AB sobre
Gold v2 ∩ cobertura. Se guardan bajo `artifacts` con su sha256 en `manifiesto_gold_b.json`. Cada fila es `{context_id, pred, probs}`:
el escritor rechaza cualquier otro campo (L7). La comparación (B − A y AB − A por pista y semilla) se hace con el Gold congelado.

In [18]:
GOLD_A_TODO = ex.casos_gold(GOLD_USO, FUENTES, PAPERS, "A")
GOLD_A_COB = ex.casos_gold(GOLD_B, FUENTES, PAPERS, "A")
GOLD_B_COB = ex.casos_gold(GOLD_B, FUENTES, PAPERS, "B", ENTRADAS_B)
GOLD_AB_COB = ex.casos_gold(GOLD_B, FUENTES, PAPERS, "AB", ENTRADAS_B)
assert ids(GOLD_A_COB) == ids(GOLD_B_COB) == ids(GOLD_AB_COB)   # conjunto pareado
PEDIDOS = {"A": {"cob": GOLD_A_COB, "todo": GOLD_A_TODO}, "B": {"cob": GOLD_B_COB}, "AB": {"cob": GOLD_AB_COB}}

MANIFIESTO_GOLD = {}
for run_id, r in LISTA_RUNS.items():
    carpeta = CORRIDAS / r["nombre"]
    if r["fase"] not in ("nucleo", "specter") or r["entrada"] not in PEDIDOS or not (carpeta / "model.safetensors").is_file():
        continue
    try:
        archivos = {}
        for etiqueta, casos in PEDIDOS[r["entrada"]].items():
            ruta = GOLD_DIR / r["nombre"] / f"pred_gold_v2_{etiqueta}.jsonl"
            if not (ruta.is_file() and len(ruta.read_text(encoding="utf-8").splitlines()) == len(casos)):
                ex.escribir_predicciones_ciegas(ruta, ex.predecir_gold(carpeta, r["encoder"], casos, r["entrada"]))
            filas = [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines()]
            assert all(set(f) == set(ex.CAMPOS_PREDICCION_CIEGA) for f in filas), "una prediccion lleva campos de mas"
            archivos[etiqueta] = {"n": len(filas), "sha256": sd.sha_archivo(ruta), "ruta": str(ruta.relative_to(REPO))}
        man = json.loads((carpeta / "manifiesto_entrenamiento.json").read_text(encoding="utf-8"))
        MANIFIESTO_GOLD[r["nombre"]] = {"run_id": run_id, "encoder": r["encoder"], "entrada": r["entrada"], "pista": r["pista"],
                                        "semilla": r["semilla"], "lr": r["lr"], "fase": r["fase"],
                                        "checkpoint_sha256": man["checkpoint_sha256"], "archivos": archivos}
        print(f"[gold] {r['nombre']}: {({e: a['n'] for e, a in archivos.items()})}")
    except Exception:
        FALLAS.append(f"gold-{r['nombre']}")
        print(f"[FALLO] gold {r['nombre']}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()
escribir_json("manifiesto_gold_b.json", {"dry": DRY, "gold_v2": len(GOLD_USO), "gold_v2_cobertura": len(GOLD_B),
                                         "ids_gold_v2_cobertura_sha256": dominio.sha_ids(GOLD_B), "runs": MANIFIESTO_GOLD})

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42917.86it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-A-M-lr2e-05-s42: {'cob': 839, 'todo': 1533}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 61053.80it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-B-M-lr2e-05-s42: {'cob': 839}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 41144.95it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-AB-M-lr2e-05-s42: {'cob': 839}


... (63 outputs left out)

[gold] specter-B-S-lr5e-05-s42: {'cob': 839}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34828.56it/s]


[gold] specter-AB-S-lr3e-05-s42: {'cob': 839}


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b_v2/manifiesto_gold_b.json')

## Fase 8 — Análisis preregistrado

1. Se repiten las comprobaciones de código del gate y se verifica que el commit del preregistro sea anterior al primer run.
2. **Primarias**: AB − A por pista con `prueba_combinada`, Holm (significancia y TOST por separado) y `veredicto`, más el IC
   jerárquico y `ic_cluster`, en `bootstrap_b.json`.
3. **Secundarias** (p sin ajustar) en `analisis_b.json`.
4. Tabla de veredictos.

In [19]:
EXP = mlflow.get_experiment_by_name(EXPERIMENTO)
RUNS_MLFLOW = mlflow.search_runs([EXP.experiment_id], output_format="list", max_results=5000) if EXP else []
if DRY:
    print("DRY: comprobaciones del gate omitidas")
else:
    comprobar_codigo(DISENO["huella_codigo"])
    inicio = min(r.info.start_time for r in RUNS_MLFLOW)
    fecha_pre = datetime.fromisoformat(GATE["fecha"]).timestamp() * 1000
    assert fecha_pre < inicio, "el commit del preregistro es posterior al primer run"
    print("código igual a HEAD y preregistro commiteado antes del primer run:", GATE["fecha"],
          datetime.fromtimestamp(inicio / 1000, timezone.utc).isoformat(timespec="seconds"))


def tabla_runs() -> pd.DataFrame:
    filas = []
    for r in RUNS_MLFLOW:
        t, m, p = r.data.tags, r.data.metrics, r.data.params
        if r.info.status != "FINISHED" or "invalido" in t or "val_f1_macro" not in m:
            continue
        filas.append({"run": t.get("mlflow.runName", "").rsplit("-", 2)[0], "pista": p.get("clave_pista"), "entrada": p.get("entrada"),
                      "encoder": p.get("encoder"), "semilla": int(p.get("semilla", -1)), "lr": float(p.get("hp_lr", "nan")),
                      "fase": t.get("fase"), "repeticion": int(t.get("repeticion", 0)),
                      **{k: m.get(k) for k in ("val_f1_macro", "val_accuracy", "val_f1_weighted", "val_mixta_f1_macro",
                                               "val_sonnet_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro", "parada_f1_macro",
                                               "vcompleto_f1_macro", "val_n", "segundos_entrenamiento", "gpu_mem_pico_gb")}})
    return pd.DataFrame(filas)


TABLA = tabla_runs()
if len(TABLA):
    LR_ELEGIDO = pd.Series([LR.get((f.pista, f.entrada)) == f.lr for f in TABLA.itertuples()], index=TABLA.index)
    NUCLEO = TABLA[(TABLA["fase"] == "nucleo") & LR_ELEGIDO]
else:
    NUCLEO = TABLA
COLS = ["val_f1_macro", "val_accuracy", "val_f1_weighted", "parada_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro"]
if len(NUCLEO):
    print(NUCLEO.groupby(["pista", "encoder", "entrada"])[COLS + ["segundos_entrenamiento"]].agg(["mean", "std", "count"]).round(4).to_string())
if len(TABLA):
    TABLA.sort_values(["pista", "encoder", "entrada", "lr", "semilla"]).to_csv(EVIDENCIAS / "resumen_b.csv", index=False)

código igual a HEAD y preregistro commiteado antes del primer run: 2026-11-14T21:54:50-05:00 2026-11-15T02:54:58+00:00
                      val_f1_macro               val_accuracy               val_f1_weighted               parada_f1_macro               val_mixta_f1_macro               val_sonnet_f1_macro               segundos_entrenamiento               
                              mean     std count         mean     std count            mean     std count            mean     std count               mean     std count                mean     std count                   mean      std count
pista encoder entrada                                                                                                                                                                                                                           
M     scibert A             0.6610  0.0111     5       0.8133  0.0064     5          0.8097  0.0067     5          0.7308  0.0063     5             0.6610  0.

In [20]:
PIDS = {}   # {(pista, entrada, semilla, encoder, conjunto, repeticion): {context_id: pred}}


def preds_de(k, entrada, s, encoder="scibert", conjunto="val", repeticion=0):
    clave = (k, entrada, s, encoder, conjunto, repeticion)
    if clave not in PIDS:
        carpeta = CORRIDAS / ex.nombre_run(encoder, entrada, k, LR[(k, LR_DE[entrada])], s, repeticion)
        PIDS[clave] = (ex.leer_predicciones(carpeta, conjunto) if (carpeta / f"predicciones_{conjunto}.jsonl").is_file()
                       else None)
    return PIDS[clave]


def alinear(k, x, a, semillas, etiquetas, ids_=None, encoder="scibert"):
    # Predicciones de X y A por semilla sobre los ids comunes (y con etiqueta), mas etiquetas y citantes
    sem = [s for s in semillas if preds_de(k, x, s, encoder) and preds_de(k, a, s, encoder)]
    if not sem:
        return None
    comunes = set(etiquetas).intersection(*[set(preds_de(k, e, s, encoder)) for s in sem for e in (x, a)])
    comunes = sorted(comunes & set(ids_) if ids_ is not None else comunes)
    if len({CTX[c]["citing_id"] for c in comunes}) < 2:
        return None
    return {"semillas": sem, "ids": comunes, "y": [etiquetas[c] for c in comunes], "g": [CTX[c]["citing_id"] for c in comunes],
            "x": [[preds_de(k, x, s, encoder)[c] for c in comunes] for s in sem],
            "a": [[preds_de(k, a, s, encoder)[c] for c in comunes] for s in sem]}


def contraste(k, x, a, semillas, etiquetas=None, ids_=None, metrica="f1_macro"):
    d = alinear(k, x, a, semillas, ETIQ_V[k] if etiquetas is None else etiquetas, ids_)
    if d is None:
        return None
    r = ex.prueba_combinada(d["x"], d["a"], d["y"], d["g"], n=N_BOOTSTRAP, semilla=SEMILLA_BOOTSTRAP, metrica=metrica, delta=DELTA)
    return {**r, "semillas_usadas": d["semillas"]}


def linea(nombre, r):
    return (f"{nombre:26s} d={r['diferencia']:+.4f} IC95=[{r['ic95'][0]:+.4f}, {r['ic95'][1]:+.4f}] p={r['p']:.4f} "
            f"d_j={[round(v, 4) for v in r['diferencia_por_semilla']]} n={r['casos']} citantes={r['grupos']}")


PRIMARIAS = {}
for k in PISTAS:
    d = alinear(k, "AB", "A", SEMILLAS_NUCLEO, ETIQ_V[k])
    if d is None:
        print(f"{k}: sin runs del núcleo")
        continue
    r = ex.prueba_combinada(d["x"], d["a"], d["y"], d["g"], n=N_BOOTSTRAP, semilla=SEMILLA_BOOTSTRAP, delta=DELTA)
    r["semillas_usadas"] = d["semillas"]
    r["ic_jerarquico"] = ex.bootstrap_jerarquico(d["x"], d["a"], d["y"], d["g"], n=N_BOOTSTRAP, semilla=SEMILLA_BOOTSTRAP)
    r["ic_cluster"] = ex.bootstrap_diferencia_cluster(d["a"], d["x"], d["y"], d["g"], n=N_BOOTSTRAP, semilla=SEMILLA_BOOTSTRAP)
    PRIMARIAS[k] = r
P_HOLM = ex.holm({k: PRIMARIAS[k]["p"] if k in PRIMARIAS else 1.0 for k in PISTAS})
P_TOST_HOLM = ex.holm({k: PRIMARIAS[k]["p_tost"] if k in PRIMARIAS else 1.0 for k in PISTAS})
VEREDICTOS = {k: ex.veredicto(PRIMARIAS[k], P_HOLM[k], P_TOST_HOLM[k], ALFA, MIN_SIGNO, len(SEMILLAS_NUCLEO))
              if k in PRIMARIAS else "incompleto" for k in PISTAS}
for k, r in PRIMARIAS.items():
    r.update({"p_holm": P_HOLM[k], "p_tost_holm": P_TOST_HOLM[k], "veredicto": VEREDICTOS[k]})
    print(f"{k} {linea('AB - A', r)} p_Holm={P_HOLM[k]:.4f} p_TOST,Holm={P_TOST_HOLM[k]:.4f} -> {VEREDICTOS[k]}")
    print(f"   IC jerárquico {tuple(round(v, 4) for v in r['ic_jerarquico']['ic95'])} | ic_cluster {tuple(round(v, 4) for v in r['ic_cluster']['ic95'])}")
escribir_json("bootstrap_b.json", {"primaria": "AB - A", "metrica": "f1_macro", "delta": DELTA, "alfa": ALFA, "min_signo": MIN_SIGNO,
                                   "semillas_nucleo": SEMILLAS_NUCLEO, "lr": {f"{k}:{b}": v for (k, b), v in LR.items()},
                                   "preregistro_commit": GATE["commit"], "pistas": PRIMARIAS, "veredictos": VEREDICTOS})

M AB - A                     d=-0.0060 IC95=[-0.0206, +0.0086] p=0.4137 d_j=[0.0007, -0.0056, 0.0029, -0.0171, -0.011] n=1719 citantes=360 p_Holm=0.8273 p_TOST,Holm=0.0602 -> no_concluyente
   IC jerárquico (-0.0225, 0.0109) | ic_cluster (-0.018, 0.007)
S AB - A                     d=-0.0062 IC95=[-0.0304, +0.0181] p=0.6122 d_j=[0.0149, 0.0004, -0.0136, -0.0104, -0.0221] n=1719 citantes=360 p_Holm=0.8273 p_TOST,Holm=0.1286 -> no_concluyente
   IC jerárquico (-0.0327, 0.0205) | ic_cluster (-0.0259, 0.0139)


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b_v2/bootstrap_b.json')

In [21]:
ANALISIS = {"contrastes": {}, "metricas_complementarias": {}, "por_clase": {}, "estratos": {}, "cruzada_maestros": {},
            "sesgo_cobertura": {}, "no_determinismo": {}, "specter": {}, "truncamiento": {}}
CONTRASTES = (("B - A", "B", "A", SEMILLAS_NUCLEO), ("AB - B", "AB", "B", SEMILLAS_NUCLEO),
              ("AB_aleatorio - AB", "AB_aleatorio", "AB", SEMILLAS_CONTROL), ("B_aleatorio - B", "B_aleatorio", "B", SEMILLAS_CONTROL),
              ("B_barajada - B", "B_barajada", "B", SEMILLAS_CONTROL), ("B_sin_seccion - B", "B_sin_seccion", "B", SEMILLAS_CONTROL),
              ("A - C0", "A", "C0", SEMILLAS_CONTROL))
for k in PISTAS:
    print(f"=== {k}: contrastes secundarios (p sin ajustar)")
    for nombre, x, a, sem in CONTRASTES:
        if (r := contraste(k, x, a, sem)) is not None:
            ANALISIS["contrastes"][f"{k}:{nombre}"] = r
            print(linea(nombre, r))
    for nombre, x, a in (("AB - A", "AB", "A"), ("B - A", "B", "A")):
        for metrica in ("accuracy", "f1_weighted"):
            if (r := contraste(k, x, a, SEMILLAS_NUCLEO, metrica=metrica)) is not None:
                ANALISIS["metricas_complementarias"][f"{k}:{nombre}:{metrica}"] = r
                print(linea(f"{nombre} [{metrica}]", r))
        if (d := alinear(k, x, a, SEMILLAS_NUCLEO, ETIQ_V[k])) is not None:
            ANALISIS["por_clase"][f"{k}:{nombre}"] = ex.bootstrap_jerarquico(d["x"], d["a"], d["y"], d["g"], n=N_BOOTSTRAP,
                                                                              semilla=SEMILLA_BOOTSTRAP)["por_clase"]
            print(f"--- {k} {nombre}: F1 por clase (media de semillas) e IC95 jerárquico")
            print(pd.DataFrame([{"clase": c["clase"], "soporte": c["soporte"], "A": c["f1_a"], "X": c["f1_x"], "X-A": c["diferencia"],
                                 "IC95 inf": c["ic95"][0], "IC95 sup": c["ic95"][1], "contrib. macro": c["contribucion_macro"]}
                                for c in ANALISIS["por_clase"][f"{k}:{nombre}"]]).round(4).to_string(index=False))
        # estratos: citado visto / no visto en el train de la pista
        for estrato, visto in (("citado_visto_en_train", True), ("citado_no_visto", False)):
            ids_e = [c for c in ETIQ_V[k] if (CTX[c]["refid"] in CITADOS_TRAIN[k]) == visto]
            if (r := contraste(k, x, a, SEMILLAS_NUCLEO, ids_=ids_e)) is not None:
                ANALISIS["estratos"][f"{k}:{nombre}:{estrato}"] = r
                print(linea(f"{nombre} [{estrato}]", r))
        # validacion cruzada entre maestros: las predicciones de la pista con la etiqueta de la otra
        otra = "S" if k == "M" else "M"
        if (r := contraste(k, x, a, SEMILLAS_NUCLEO, etiquetas=ETIQ_V[otra])) is not None:
            ANALISIS["cruzada_maestros"][f"{k}:{nombre}:etiqueta_{otra}"] = r
            print(linea(f"{nombre} [etiqueta {otra}]", r))

# Sesgo de cobertura: A del nucleo sobre V completo frente a V_B, y los ganadores de #104 en ambos
for k in PISTAS:
    vb = [f1_en(p, ETIQ_V[k])[0] for s in SEMILLAS_NUCLEO if (p := preds_de(k, "A", s))]
    vc = [f1_en(p, ETIQ_VC[k])[0] for s in SEMILLAS_NUCLEO if (p := preds_de(k, "A", s, conjunto="vcompleto"))]
    ref = [v for n, v in REFERENCIA.items() if n.startswith(f"{k}-")]
    ANALISIS["sesgo_cobertura"][k] = {
        "A_nucleo_V_B": {"media": sum(vb) / len(vb) if vb else None, "n_semillas": len(vb)},
        "A_nucleo_V_completo": {"media": sum(vc) / len(vc) if vc else None, "n_semillas": len(vc)},
        "referencia_104_V_B": {"media": sum(r["f1_macro_V_B"] for r in ref) / len(ref) if ref else None, "n_semillas": len(ref)},
        "referencia_104_V_completo": {"media": sum(r["f1_macro_V"] for r in ref) / len(ref) if ref else None, "n_semillas": len(ref)}}
    print(f"sesgo de cobertura {k}:", {n: (round(v["media"], 4) if v["media"] is not None else None) for n, v in ANALISIS["sesgo_cobertura"][k].items()})

# No determinismo: A-M-s42 repetido
p0, p1 = preds_de("M", "A", 42), preds_de("M", "A", 42, repeticion=1)
if p0 and p1:
    comunes = sorted(set(p0) & set(p1) & set(ETIQ_V["M"]))
    f0, f1 = f1_en(p0, ETIQ_V["M"])[0], f1_en(p1, ETIQ_V["M"])[0]
    ANALISIS["no_determinismo"] = {"casos": len(comunes), "concordancia": sum(p0[c] == p1[c] for c in comunes) / len(comunes),
                                   "f1_original": f0, "f1_repeticion": f1, "abs_dif_f1": abs(f1 - f0)}
    print("no determinismo:", ANALISIS["no_determinismo"])

# SPECTER (descriptivo, s42)
for k in PISTAS:
    f1s = {e: f1_en(p, ETIQ_V[k])[0] for e in BRAZOS_GRILLA if (p := preds_de(k, e, 42, encoder="specter"))}
    if f1s:
        ANALISIS["specter"][k] = {**f1s, **({"AB - A": f1s["AB"] - f1s["A"]} if {"A", "AB"} <= set(f1s) else {}),
                                  **({"B - A": f1s["B"] - f1s["A"]} if {"A", "B"} <= set(f1s) else {})}
        print(f"SPECTER {k}:", {n: round(v, 4) for n, v in ANALISIS["specter"][k].items()})

# Truncamiento (SciBERT): fraccion de casos de V_B con recorte por segmento
for k in PISTAS:
    for entrada in ("A", "B", "AB", "C0"):
        res = ct.resumen_truncamiento(ejemplos(entrada, V_B[k]))
        ANALISIS["truncamiento"][f"{k}:{entrada}"] = res
        print(f"{k} {entrada}: longitud media {res['longitud_media']:.0f} | " +
              ", ".join(f"{s}: {v['frac_truncado']:.2f}" for s, v in res["segmentos"].items()))
ANALISIS.update({"referencia_104": REFERENCIA, "cobertura": COBERTURA, "aleatorias": RESUMEN_ALEATORIAS,
                 "seleccion_lr": {f"{k}:{b}": v for (k, b), v in LR.items()}})
escribir_json("analisis_b.json", ANALISIS)

DF = rm.exportar_manifiesto(EVIDENCIAS, EXPERIMENTO)
print(len(DF), "runs exportados a", EVIDENCIAS.relative_to(REPO))
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

=== M: contrastes secundarios (p sin ajustar)
B - A                      d=-0.0111 IC95=[-0.0312, +0.0090] p=0.2675 d_j=[-0.004, -0.0095, 0.002, -0.0339, -0.0101] n=1719 citantes=360
AB - B                     d=+0.0051 IC95=[-0.0088, +0.0190] p=0.4707 d_j=[0.0047, 0.0039, 0.0008, 0.0168, -0.0008] n=1719 citantes=360
AB_aleatorio - AB          d=+0.0058 IC95=[-0.0063, +0.0179] p=0.3459 d_j=[0.0079, 0.0047, 0.0048] n=1719 citantes=360
B_aleatorio - B            d=+0.0030 IC95=[-0.0177, +0.0236] p=0.7640 d_j=[0.0138, 0.0009, -0.0058] n=1719 citantes=360
B_barajada - B             d=-0.0037 IC95=[-0.0170, +0.0097] p=0.5886 d_j=[-0.0046, -0.0057, -0.0008] n=1719 citantes=360
B_sin_seccion - B          d=-0.0015 IC95=[-0.0325, +0.0294] p=0.9094 d_j=[-0.0198, 0.0126, 0.0026] n=1719 citantes=360
A - C0                     d=-0.0091 IC95=[-0.0378, +0.0196] p=0.4789 d_j=[-0.0202, -0.0155, 0.0084] n=1719 citantes=360
AB - A [accuracy]          d=-0.0015 IC95=[-0.0098, +0.0067] p=0.7124 d_j=[0.00

In [22]:
VEREDICTO_TABLA = pd.DataFrame([
    {"pista": k, "estimando": "AB - A", "semillas": len(PRIMARIAS[k]["semillas_usadas"]) if k in PRIMARIAS else 0,
     "d": PRIMARIAS[k]["diferencia"] if k in PRIMARIAS else None,
     "IC95 inf": PRIMARIAS[k]["ic95"][0] if k in PRIMARIAS else None, "IC95 sup": PRIMARIAS[k]["ic95"][1] if k in PRIMARIAS else None,
     "p": PRIMARIAS[k]["p"] if k in PRIMARIAS else None, "p_Holm": P_HOLM[k],
     "p_TOST": PRIMARIAS[k]["p_tost"] if k in PRIMARIAS else None, "p_TOST_Holm": P_TOST_HOLM[k],
     "d_j con el signo de d": (sum(1 for v in PRIMARIAS[k]["diferencia_por_semilla"]
                                  if v * PRIMARIAS[k]["diferencia"] > 0) if k in PRIMARIAS else None),
     "veredicto": VEREDICTOS[k]} for k in PISTAS])
print(f"Veredictos preregistrados (δ = {DELTA}, α = {ALFA}, F1-macro sobre V_B)")
print(VEREDICTO_TABLA.round(4).to_string(index=False))

Veredictos preregistrados (δ = 0.02, α = 0.05, F1-macro sobre V_B)
pista estimando  semillas       d  IC95 inf  IC95 sup      p  p_Holm  p_TOST  p_TOST_Holm  d_j con el signo de d      veredicto
    M    AB - A         5 -0.0060   -0.0206    0.0086 0.4137  0.8273  0.0301       0.0602                      3 no_concluyente
    S    AB - A         5 -0.0062   -0.0304    0.0181 0.6122  0.8273  0.1286       0.1286                      3 no_concluyente
